# Measuring tune response ophyd async style

## import and set up

###  standard modules

In [ ]:
import logging


import yaml
import jsons
import pprint

### logging level

In [ ]:
# avoid being messed up with communication debug messages
logging.basicConfig(level=logging.WARNING)


### accml and accml lib modules

In [ ]:
from accml.core.utils.basic_measurement_execution_engine import BasicMeasurementExecutionEngine
from accml.core.utils.simple_storage import SimpleDataStorage
from accml.custom.ophyd_async.ophyd_async_backend import OphydAsyncDeviceBackendRW
from accml.custom.ophyd_async.ophyd_async_measurement_execution_engine import OphydAsyncDeltaBackendRWProxy
from accml_lib.core.bl.command_rewritter import CommandRewriter
from accml_lib.core.bl.delta_backend import StateCache
from accml_lib.core.model.output.tune import Tune
from accml_lib.custom.bessyii.liasion_translator_setup import load_managers

from accml.app.tune.model import TuneResponseCollection

In [ ]:
from accml_lib.core.model.utils.command import Command, CommandSequence, ReadCommand, TransactionCommand, BehaviourOnError

### jsons for exporting data model

jsons takes care to export the datamodel into basic types (lists, dictonaries, int, string, float...)
the output can then be stored in appropriate formates eg. json or yaml

In [ ]:
from accml_lib.core.model.output.result import register_serializers_to_json_fork

jsons_fork = jsons.fork()
register_serializers_to_json_fork(jsons_fork)

## device setup and connection

### connection to devices

In [ ]:
from accml_lib.custom.bessyii.setup import setup

devices = setup(prefix="")

await devices.get("tune").connect()
await devices.get('quadrupole_pcs').connect()

### setup of communication

In [ ]:
state_cache=StateCache(name="BESSY2_OphydAsync_Dev_State_Cache")
backend = OphydAsyncDeviceBackendRW(devices=devices)
delta_backend = OphydAsyncDeltaBackendRWProxy(
    backend=backend,
    state_cache,
)
mexec = BasicMeasurementExecutionEngine(
    backend=delta_backend,
    cmd_rewriter=CommandRewriter(liaison_manager=lm, translation_service=ts),
    storage=SimpleDataStorage(),
    expected_view_for_output="device"
)


## The measurement itself

### The commands

In [ ]:
measurement_values = [0, -1, 0, 1, 0]

In [ ]:
cmds_on_machine = []
for name in yp.get("quadrupoles"):
    for val in measurement_values:
        cmds_on_machine.append(
            TransactionCommand(
                transaction=[Command(id=name, property="delta_set_current", value=val, behaviour_on_error=BehaviourOnError.stop)]
            )
        )
cmds_on_machine = CommandSequence(commands=cmds_on_machine)
pprint.pprint(cmds_on_machine)

### Take the measurement

In [ ]:
from typing import Sequence

detectors = [ReadCommand("tune", "transversal")]

In [ ]:
md = {}

uid = mexec.execute(
    detectors=detectors,
    commands_collection=cmds_on_machine.commands,  # need to add bpms
    # detectors=[tunes],
#    # actuators=actuators,
    # info_signals=info_signals,
    md=md,
)

uid,

### Analysis preparation: what happened  

#### reference point: inspect cache

In [ ]:
list(state_cache.cache)

In [ ]:
state_cache.get(ReadCommand(id="Q1M1D1R", property="set_current"))

In [ ]:
data = storage.get(uuid)
data = jsons.dump(data, fork_inst=jsons_fork)

with open("tune_response_data_from_simulator.json", "wt") as fp:
    json.dump(data, fp, indent=4)

